In [133]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

In [134]:
dog_imgs = Path('../datasets/cat-dog-images/dog-set')
cat_imgs = Path('../datasets/cat-dog-images/cat-set')

dogs = []
cats = []

for img in dog_imgs.iterdir():
    _img = Image.open(img)
    _img = _img.resize((64, 64))
    grey = _img.convert('L')
    
    dogs.append(np.array(grey))

for img in cat_imgs.iterdir():
    _img = Image.open(img)
    _img = _img.resize((64, 64))
    grey = _img.convert('L')

    cats.append(np.array(grey))

dogs = np.array(dogs)
cats = np.array(cats)

dogs = dogs / 255
cats = cats / 255

X = np.vstack((dogs, cats))
y = np.concatenate((np.zeros(dogs.shape[0], dtype=int), np.ones(cats.shape[0], dtype=int)))

In [135]:
# Randomize data and target 

perm = np.random.permutation(X.shape[0])

X = X[perm]
y = y[perm]

In [136]:
# Split the data into training and testing

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [137]:
# Flatten the dataset while preserving the row number

X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)

In [138]:
# Define the functions

def relu(x):
    return np.maximum(0, x)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu_derivative(x):
    return (x > 0).astype(float)

In [139]:
# Define the ANN

class HiddenLayer():
    def __init__(self, x1, x2, act_func):
        self.weights = np.random.randn(x1, x2) * np.sqrt(2.0 / x1)
        self.bias = np.zeros(x2)
        self.act_func = act_func

    def forward_pass(self, x):
        z = np.dot(x, self.weights) + self.bias
        y = self.act_func(z)
        
        return x, z, y

class LayerData():
    def __init__(self, inputs, inter, outputs):
        self.inputs = inputs
        self.inter = inter
        self.outputs = outputs

class Classifier():
    def __init__(self):
        self.layer1 = HiddenLayer(4096, 512, relu)
        self.layer2 = HiddenLayer(512, 256, relu)
        self.layer3 = HiddenLayer(256, 128, relu)
        self.layer4 = HiddenLayer(128, 16, relu)
        self.layer5 = HiddenLayer(16, 1, sigmoid)

    def get_layers(self):
        return [self.layer1, self.layer2, self.layer3, self.layer4, self.layer5]

    def forward(self, x):
        inputs = []
        inter = []
        outputs = []

        x, z, res = self.layer1.forward_pass(x)
        inputs.append(x)
        inter.append(z)
        outputs.append(res)

        x, z, res = self.layer2.forward_pass(res)
        inputs.append(x)
        inter.append(z)
        outputs.append(res)

        x, z, res = self.layer3.forward_pass(res)
        inputs.append(x)
        inter.append(z)
        outputs.append(res)

        x, z, res = self.layer4.forward_pass(res)
        inputs.append(x)
        inter.append(z)
        outputs.append(res)

        x, z, res = self.layer5.forward_pass(res)
        inputs.append(x)
        inter.append(z)
        outputs.append(res)

        return res, LayerData(inputs, inter, outputs)
        

In [140]:
# Define the model

model = Classifier()

In [141]:
# Define the loss function

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)

    return -(y_true * np.log(y_pred) +
             (1 - y_true) * np.log(1 - y_pred))

In [142]:
# Train the model

# Schocastically (for now)
def train_model(model, X, y, epochs, learning_rate):
    for _ in range(epochs):
        for _x, _y in zip(X, y):
            y_hat, layer_data = model.forward(_x)

            loss = binary_cross_entropy(_y, y_hat)

            reversed_layers = model.get_layers()[::-1]

            for idx, layer in enumerate(reversed_layers):
                data_idx = len(reversed_layers) - idx - 1

                if idx == 0:
                    delta = y_hat - _y
                    
                dW = np.outer(layer_data.inputs[data_idx], delta)
                dB = delta
                
                if idx != len(reversed_layers) - 1:
                    next_delta = (delta @ layer.weights.T) * relu_derivative(
                        layer_data.inter[data_idx - 1]
                    )
                    delta = next_delta
                
                layer.weights -= learning_rate * dW
                layer.bias -= learning_rate * dB

        print(f"Current loss: {loss}\n")

In [143]:
# Train the model

train_model(model=model, X=X_train, y=y_train, epochs=10, learning_rate=0.001)

Current loss: [0.74199871]

Current loss: [0.66020155]

Current loss: [0.74290303]

Current loss: [0.60781017]

Current loss: [0.550072]

Current loss: [0.54706484]

Current loss: [0.51931506]

Current loss: [0.45134076]

Current loss: [0.4570102]

Current loss: [0.33620781]



In [144]:
# Classify the images 

y_pred, _ = model.forward(X_test)
y_pred = np.where(y_pred >= 0.5, 1, 0)

In [145]:
# Get the accuracy score and the confusion matrix

from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

0.575